# CrackSense AI - Phase 1: Dataset Exploration & Leakage-Aware Splitting

### Problem Overview:
- **Dataset**: 40,000 concrete image patches ($227 \times 227$ RGB).
- **Classes**: 20,000 Positive (Cracked) and 20,000 Negative (Clear).
- **Data Leakage Risk**: The 40,000 patches were cropped from only **458 original photographs**. If patches from the same photo end up in both training and testing sets, the model achieves falsely inflated accuracy by memorizing wall aggregate patterns rather than learning true crack features.
- **Solution**: Group sibling patches using **Perceptual Difference Hashing (dHash)** into clusters, and assign entire clusters exclusively to Train (70%), Validation (15%), or Test (15%).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai.preprocessing.explore_and_split import run_phase1_pipeline, DatasetAuditor, compute_dhash, hamming_distance

## 1. Scan and Audit Dataset Health
Check image counts, resolutions, formats, and corruptions in `dataset/raw/`.

In [ ]:
auditor = DatasetAuditor("../../dataset/raw")
pos_files, neg_files = auditor.scan_files()
print(f"Positive images found: {len(pos_files)}")
print(f"Negative images found: {len(neg_files)}")

# Plot Class Balance
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Positive (Crack)", "Negative (Clear)"], [len(pos_files), len(neg_files)], color=["#ef4444", "#22c55e"])
ax.set_ylabel("Number of Images")
ax.set_title("CrackSense AI - Class Balance")
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:,}", ha='center', va='bottom', fontweight='bold')
plt.show()

## 2. Visualize Sample Concrete Surface Patches

In [ ]:
def show_samples(files, title, count=4):
    fig, axes = plt.subplots(1, count, figsize=(12, 3))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i in range(min(count, len(files))):
        img = Image.open(files[i])
        axes[i].imshow(img)
        axes[i].set_title(f"{img.width}x{img.height}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

if pos_files:
    show_samples(pos_files, "Positive Samples (Cracked Concrete)")
if neg_files:
    show_samples(neg_files, "Negative Samples (Clear Concrete)")

## 3. Execute Leakage-Aware Splitting
Clusters sibling patches via perceptual hash similarity and produces `train.csv`, `val.csv`, and `test.csv`.

In [ ]:
summary = run_phase1_pipeline(
    raw_dir="../../dataset/raw",
    output_dir="../../dataset/processed",
    hamming_threshold=6
)
print("Split successfully created with zero leakage! Summary:", summary)